# 字符串的模式匹配

In [1]:
import re; 

In [2]:
motto = "Verba volant, scripta manent"; 

In [3]:
#构造Unicode从U+0000到U+FFFF的所有字符顺次连接组成的字符串
char = str().join( 
    chr(x) for x in range(65536)
); 

在`ipython`命令行或者`Jupyter notebook`中, 我们可以直接使用特定的转义字符, 改变打印到`sys.stdout`或者笔记本输出单元的字符样式. 本文档据此构造函数, 实现正则表达式匹配内容标记功能. 

In [4]:
#构造生成器, 用于从可迭代对象中有重叠顺次获取相邻两个元素
import copy;  
def adjecent(iter_0): 
    iter_1 = copy.copy(iter_0).__iter__(); 
    iter_2 = copy.copy(iter_0).__iter__(); 
    iter_2.__next__(); 
    for elem_2 in iter_2: 
        elem_1 = iter_1.__next__(); 
        yield elem_1, elem_2; 

In [5]:
#计算正则表达式regex在字符串text中匹配所得子串的起止位置
def pattern_position(text, regex): 
    indices = tuple(
        match.span() for match in re.compile(regex).finditer(text)
    ); 
    return(indices); 

In [6]:
#将正则表达式regex在字符串text中匹配所得子串分布范围使用转义序列标识, 
#以便使用io.TextIOWrapper.write方法打印到stdout或笔记本输出单元
def pattern_highlight(text, regex): 
    indices = pattern_position(text, regex); 
    text_disp = str(); 
    substr_stat = bytearray(len(text) + 1); 
    for idx in indices: 
        start, end = idx; 
        substr_stat[start + 1: end + 1] = (1, ) * (end - start); 
    substr_stat = bytes(substr_stat); 
    for (ch, (start, end)) in zip("\x00" + text, adjecent(substr_stat)): 
        text_disp += ch; 
        if not bool(start) and bool(end): 
            text_disp += "\x1b[07m"; 
        elif bool(start) and not bool(end): 
            text_disp += "\x1b[0m"; 
    text_disp = "\x1b[0m{raw:s}{end:s}\x1b[0m".format(
        raw=text_disp[1:], end=text[-1]
    ); 
    return(text_disp); 

In [7]:
#正则表达式及其匹配结果的对比显示
def pattern_match_illustrate(text, patterns): 
    for regex in [str(), ] + patterns: 
        print("{regex:\x20<24s}{disp:<s}".format(
            regex=regex[: 23], disp=pattern_highlight(text, regex)
        ) )

## 正则表达式语法

### 单字符匹配语法
|模式|功能|备注|
|:-|:-:|:-|
|`a`|字面意义上匹配普通字符`a`|以下字符需要使用反斜杠(`\`)转义: <br>`(`, `)`, `[`, `]`, `\`, `.`, `^`, `$`, <br>`*`, `+`, `?`, `\|`|
|`.`|匹配除`\n`(换行符)以外的<br>任何字符|当启用`re.DOTALL`时, 解除对<br>`\n`的匹配限制|
|`[abc]`|匹配`a`, `b`, `c`之一|当该语法在同一正则表达式中<br>多次出现时, 不同位置的匹配是<br>相互独立的, 例如`[ab][ab]`<br>可以匹配`aa`, `ab`, `ba`或`bb`; <br>在方括号中, `-`  (半角负号) 作为<br>待匹配的字符时, 需要转义为`\-`|
|`[^abc]`|匹配除`a`, `b`, `c`以外的任何字符|当该语法在同一正则表达式中<br>多次出现时, 不同位置的匹配是<br>相互独立的|
|`[a-z]`|匹配字符编码大于等于`a`且<br>小于等于`z`的字符||
|`\w`|匹配`_` (半角下划线) , 或者<br>调用`isalnum`方法返回<br>`True`的单个字符||
|`\W`|匹配调用`isalnum`方法返回<br>`False`的单个字符, 不包括`_`||
|`\d`|匹配调用`isdecimal`方法返回<br>`True`的单个字符||
|`\D`|匹配调用`isdecimal`方法返回<br>`False`的单个字符||

In [8]:
tp_regex = [
    r"a", r".", r"[aeiou]", r"[^aeiou]", r"[p-t]", r"\w", r"\d"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
a                       Verba volant, scripta manent
.                       Verba volant, scripta manent
[aeiou]                 Verba volant, scripta manent
[^aeiou]                Verba volant, scripta manent
[p-t]                   Verba volant, scripta manent
\w                      Verba volant, scripta manent
\d                      Verba volant, scripta manent


#### `\w`, `\d`的匹配范围演示

In [9]:
#表示文本或者数目的字符
str_alnum = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isalnum()]
); 

In [10]:
#正则表达式\w语法可以匹配的字符
re_mtch_word = str().join(re.compile("\w").findall(char)); 

In [11]:
set(re_mtch_word).issuperset(set(str_alnum))

True

In [12]:
#正则表达式\w语法可以匹配的字符, 比满足isalnum的字符范围多一个_ (半角下划线)
set(re_mtch_word).difference(set(str_alnum))

{'_'}

In [13]:
#可用于十进制数码的字符
str_dec = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isdecimal()]
); 

In [14]:
#正则表达式\d语法可以匹配的字符
re_mtch_dec = str().join(re.compile("\d").findall(char)); 

In [15]:
set(re_mtch_dec) == set(str_dec)

True